# Colab runtime setup

Upload this notebook to Colab, select Python 3 and an available GPU in Runtime → Change runtime type, then connect. Run each code cell individually. This notebook inspects the supplied runtime; it installs no packages and performs no model training. Keep saved notebooks and results private.


In [ ]:
import json
import os
import platform
from pathlib import Path
import shutil
import subprocess
from datetime import datetime, timezone

import torch

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
report_dir = Path("/content/posttraining-results/colab-setup") / run_id
report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / "environment.json"
cuda = torch.cuda.is_available()
bf16 = bool(cuda and torch.cuda.is_bf16_supported(including_emulation=False))
dtype_name = "bfloat16" if bf16 else "float16" if cuda else "float32"
report = {
    "timestamp_utc": run_id,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda_build": torch.version.cuda,
    "cuda_available": cuda,
    "bf16_supported": bf16,
    "bf16_check_including_emulation": False,
    "selected_dtype": dtype_name,
    "ram_total_gib": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30, 2),
    "runtime_disk_free_gib": round(shutil.disk_usage("/content").free / 2**30, 2),
    "devices": [
        {"index": i, "name": torch.cuda.get_device_name(i),
         "vram_gib": round(torch.cuda.get_device_properties(i).total_memory / 2**30, 2),
         "capability": list(torch.cuda.get_device_capability(i))}
        for i in range(torch.cuda.device_count())
    ],
    "scope": "Preinstalled runtime probe; pinned project environment and CPT step NOT VERIFIED",
}
if shutil.which("nvidia-smi"):
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30)
    report["nvidia_smi"] = {"exit_code": result.returncode,
                           "stdout": result.stdout, "stderr": result.stderr}
if cuda:
    try:
        value = torch.ones((8, 8), device="cuda", dtype=getattr(torch, dtype_name))
        report["finite_gpu_matmul"] = bool(torch.isfinite(value @ value).all().item())
        del value
    except Exception as exc:
        report["finite_gpu_matmul"] = False
        report["gpu_error"] = str(exc)
else:
    report["finite_gpu_matmul"] = False
report_path.write_text(json.dumps(report, indent=2) + "\n")
print(report_path)
print(json.dumps(report, indent=2))
assert cuda, "No GPU is available. Check the assigned runtime or return Colab's allocation error."
assert report["finite_gpu_matmul"], "GPU tensor check failed; return the saved report."


## Optional: persist the report in Google Drive

Run this cell if you want a persistent Drive copy. Review Google's authorization dialog and select the intended account. Mounting Drive grants notebook code access to Drive files; it is not limited to the output folder. If you prefer not to mount Drive, skip this cell and download the report below.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
drive_report_dir = Path("/content/drive/MyDrive/posttraining-results/colab-setup") / run_id
drive_report_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(report_path, drive_report_dir / report_path.name)
print("Persistent report:", drive_report_dir / report_path.name)


## Download the report

Download this JSON report even if the first cell reported a GPU failure. Return it to your local private results directory. The original notebook has no saved outputs; download the executed notebook separately from Colab's File menu if a full execution record is needed.


In [ ]:
from google.colab import files

files.download(str(report_path))
